In [4]:
!pip install transformers torch datasets -q



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\rautr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


In [6]:
# Load FINAL dataset (CORRECT)
df = pd.read_csv("../data/processed/final_emails.csv")

print(df.shape)
print(df["label"].value_counts())
df.head()


C:\Users\rautr\AppData\Local\Temp\ipykernel_16776\2569813424.py:2: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/final_emails.csv")


(2821401, 5)
label
request      2811774
spam            5172
complaint       4360
feedback          95
Name: count, dtype: int64


,text,label,idx,Title,Labels
0,The Summer of XX/XX/2018 I was denied a mortga...,complaint,NaN,NaN,NaN
1,There are many mistakes appear in my report wi...,complaint,NaN,NaN,NaN
2,There are many mistakes appear in my report wi...,complaint,NaN,NaN,NaN
3,There are many mistakes appear in my report wi...,complaint,NaN,NaN,NaN
4,There are many mistakes appear in my report wi...,complaint,NaN,NaN,NaN


In [7]:
# Keep only required columns
df = df[["text", "label"]]

df.head()


,text,label
0,The Summer of XX/XX/2018 I was denied a mortga...,complaint
1,There are many mistakes appear in my report wi...,complaint
2,There are many mistakes appear in my report wi...,complaint
3,There are many mistakes appear in my report wi...,complaint
4,There are many mistakes appear in my report wi...,complaint


In [8]:
# 🔥 FIX tokenizer error: clean text column
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

print("After cleaning:", df.shape)
df.head()


After cleaning: (2821400, 2)


,text,label
0,The Summer of XX/XX/2018 I was denied a mortga...,complaint
1,There are many mistakes appear in my report wi...,complaint
2,There are many mistakes appear in my report wi...,complaint
3,There are many mistakes appear in my report wi...,complaint
4,There are many mistakes appear in my report wi...,complaint


In [9]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

print(label_encoder.classes_)
df.head()


['complaint' 'feedback' 'request' 'spam']


,text,label,label_id
0,The Summer of XX/XX/2018 I was denied a mortga...,complaint,0
1,There are many mistakes appear in my report wi...,complaint,0
2,There are many mistakes appear in my report wi...,complaint,0
3,There are many mistakes appear in my report wi...,complaint,0
4,There are many mistakes appear in my report wi...,complaint,0


In [10]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label_id"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))


Training samples: 2257120
Validation samples: 564280


In [11]:
# 🔥 LIMIT DATA to avoid kernel crash (VERY IMPORTANT)

MAX_TRAIN = 20000
MAX_VAL = 5000

train_texts = train_texts[:MAX_TRAIN]
train_labels = train_labels[:MAX_TRAIN]

val_texts = val_texts[:MAX_VAL]
val_labels = val_labels[:MAX_VAL]

print("Reduced training samples:", len(train_texts))
print("Reduced validation samples:", len(val_texts))


Reduced training samples: 20000
Reduced validation samples: 5000


In [12]:
# Text safety before tokenization
train_texts = [str(t) for t in train_texts]
val_texts = [str(t) for t in val_texts]

print("Text cleaning done")
print("Sample train text:", train_texts[0][:100])


Text cleaning done
Sample train text: @520507 Saw your tweet. Can you share more details and specify the branch location? ^FX


In [13]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)


In [14]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding="max_length",
    max_length=128
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding="max_length",
    max_length=128
)

print("Tokenization completed")


Tokenization completed


In [15]:
print(type(train_labels))
print(type(val_labels))
print(train_labels[:10])


<class 'list'>
<class 'list'>
[2, 2, 2, 2, 2, 2, 2, 3, 2, 0]


In [16]:
import torch

class EmailDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [17]:
train_dataset = EmailDataset(train_encodings, train_labels)
val_dataset = EmailDataset(val_encodings, val_labels)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))


Train dataset size: 20000
Validation dataset size: 5000


In [18]:
from transformers import DistilBertForSequenceClassification

NUM_CLASSES = len(set(train_labels))  # auto detect classes

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_CLASSES
)

print("Model loaded with classes:", NUM_CLASSES)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded with classes: 4


In [19]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none"
)


In [20]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [21]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Trainer initialized")



Trainer initialized


In [49]:
trainer.train()


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,0.046800
1000,0.003700
1500,0.000300
2000,0.000100
2500,0.000100
3000,0.000000
3500,0.000000
4000,0.002500
4500,0.000000
5000,0.000000


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=7500, training_loss=0.003694917605196436, metrics={'train_runtime': 16867.1682, 'train_samples_per_second': 3.557, 'train_steps_per_second': 0.445, 'total_flos': 1987081850880000.0, 'train_loss': 0.003694917605196436, 'epoch': 3.0})

In [22]:
metrics = trainer.evaluate()
print(metrics)


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 1.5549020767211914, 'eval_model_preparation_time': 0.003, 'eval_accuracy': 0.0018, 'eval_precision': 1.9447779111644658e-05, 'eval_recall': 0.0018, 'eval_f1': 3.8479809976247026e-05, 'eval_runtime': 354.5102, 'eval_samples_per_second': 14.104, 'eval_steps_per_second': 1.763}


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [31]:
trainer.save_model("distilbert_email_classifier")
tokenizer.save_pretrained("distilbert_email_classifier")


('distilbert_email_classifier\\tokenizer_config.json',
 'distilbert_email_classifier\\special_tokens_map.json',
 'distilbert_email_classifier\\vocab.txt',
 'distilbert_email_classifier\\added_tokens.json',
 'distilbert_email_classifier\\tokenizer.json')

In [30]:
trainer.state


TrainerState(epoch=None, global_step=0, max_steps=0, logging_steps=500, eval_steps=500, save_steps=500, train_batch_size=None, num_train_epochs=0, num_input_tokens_seen=0, total_flos=0, log_history=[{'eval_loss': 1.5549020767211914, 'eval_model_preparation_time': 0.003, 'eval_accuracy': 0.0018, 'eval_precision': 1.9447779111644658e-05, 'eval_recall': 0.0018, 'eval_f1': 3.8479809976247026e-05, 'eval_runtime': 354.5102, 'eval_samples_per_second': 14.104, 'eval_steps_per_second': 1.763, 'step': 0}], best_metric=None, best_global_step=None, best_model_checkpoint=None, is_local_process_zero=True, is_world_process_zero=True, is_hyper_param_search=False, trial_name=None, trial_params=None, stateful_callbacks={'TrainerControl': {'args': {'should_training_stop': False, 'should_epoch_stop': False, 'should_save': False, 'should_evaluate': False, 'should_log': False}, 'attributes': {}}})

In [1]:
df['label'].value_counts()


NameError: name 'df' is not defined

In [ ]:
# train_encodings = tokenizer(
#     train_texts,
#     truncation=True,
#     padding=True,
#     max_length=128
# )

# val_encodings = tokenizer(
#     val_texts,
#     truncation=True,
#     padding=True,
#     max_length=128
# )

# print("Tokenization done")


: 

In [ ]:
# import pandas as pd
# from sklearn.preprocessing import LabelEncoder

# # Load cleaned dataset
# df = pd.read_csv("../data/processed/cleaned_complaints.csv")

# # Keep only required columns
# df = df[["clean_text", "category"]]

# # Remove missing values (safety)
# df = df.dropna()

# # Encode labels
# label_encoder = LabelEncoder()
# df["label"] = label_encoder.fit_transform(df["category"])

# # Check data
# print(df.head())
# print("Number of classes:", len(label_encoder.classes_))


                                          clean_text  \
0  summer xx xx denied mortgage loan due charge x...   
1  many mistakes appear report without understanding   
2  many mistakes appear report without understanding   
3  many mistakes appear report without understanding   
4  many mistakes appear report without understanding   

                                            category  label  
0  Credit reporting, credit repair services, or o...      2  
1  Credit reporting, credit repair services, or o...      2  
2  Credit reporting, credit repair services, or o...      2  
3  Credit reporting, credit repair services, or o...      2  
4  Credit reporting, credit repair services, or o...      2  
Number of classes: 9


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["clean_text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))


Training samples: 3488
Validation samples: 872


In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization completed")


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenization completed


In [ ]:
import torch
from torch.utils.data import Dataset

class EmailDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = EmailDataset(train_encodings, train_labels)
val_dataset = EmailDataset(val_encodings, val_labels)

print("Datasets ready for DistilBERT training")


Datasets ready for DistilBERT training


In [ ]:
from transformers import DistilBertForSequenceClassification

num_labels = len(set(train_labels))

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

print("DistilBERT model loaded with", num_labels, "labels")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBERT model loaded with 9 labels


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Trainer setup completed")


Trainer setup completed


In [ ]:
trainer.train()


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,1.739400
100,1.602300
150,1.430200
200,1.428100
250,1.265000
300,1.211000
350,1.041900
400,0.928600
450,0.935800
500,0.882400


TrainOutput(global_step=872, training_loss=1.0364843650695381, metrics={'train_runtime': 1873.6012, 'train_samples_per_second': 3.723, 'train_steps_per_second': 0.465, 'total_flos': 231051983044608.0, 'train_loss': 1.0364843650695381, 'epoch': 2.0})

In [ ]:
metrics = trainer.evaluate()
print(metrics)


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.7637556791305542, 'eval_runtime': 66.1637, 'eval_samples_per_second': 13.179, 'eval_steps_per_second': 1.647, 'epoch': 2.0}


In [ ]:
trainer.save_model("./distilbert_email_classifier")
tokenizer.save_pretrained("./distilbert_email_classifier")

print("Final model saved successfully")


Final model saved successfully


In [ ]:
from transformers import DistilBertTokenizerFast
import torch

# Load DistilBERT tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

# Tokenize text
encodings = tokenizer(
    df["clean_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

# Convert labels to tensor
labels = torch.tensor(df["label"].values)

print("Tokenization completed")
print("Sample input_ids length:", len(encodings["input_ids"][0]))


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rautr\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingfa

Tokenization completed
Sample input_ids length: 128


In [ ]:
from sklearn.model_selection import train_test_split

# Train-test split (BERT style)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["clean_text"].tolist(),
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))


Training samples: 3488
Validation samples: 872


In [ ]:
# Tokenize train and validation text
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=128
)

print("Train & Validation tokenization done")


Train & Validation tokenization done


In [ ]:
from torch.utils.data import Dataset

class EmailDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmailDataset(train_encodings, train_labels)
val_dataset = EmailDataset(val_encodings, val_labels)

print("Dataset ready for DistilBERT training")


Dataset ready for DistilBERT training


In [ ]:
from transformers import DistilBertForSequenceClassification

num_labels = len(set(train_labels))

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

print("DistilBERT model loaded with", num_labels, "labels")


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBERT model loaded with 3488 labels


In [ ]:
!pip install -U accelerate transformers torch



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\rautr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import torch
import transformers
import accelerate

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.9.1+cpu
Transformers: 4.57.3
Accelerate: 1.12.0


In [ ]:
import os

RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Results folder ready:", RESULTS_DIR)


Results folder ready: ../results


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["clean_text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))


NameError: name 'df' is not defined